In [20]:
import pandas as pd
import time
import os

In [21]:
file_path = r'read_optimize.csv'

start_time = time.time()
df_basic = pd.read_csv(file_path, encoding='utf-8')
basic_time = time.time() - start_time

basic_memory = df_basic.memory_usage(deep=True).sum() / 1024 / 1024

In [22]:
print("1. 基础读取（无优化）：")
print(f"读取耗时：{basic_time:.4f} 秒")
print(f"内存占用：{basic_memory:.4f} MB")
print(f"文件大小：{os.path.getsize(file_path)/1024:.2f} KB")

1. 基础读取（无优化）：
读取耗时：0.0037 秒
内存占用：0.5060 MB
文件大小：68.45 KB


In [23]:
start_time = time.time()
df_cols = pd.read_csv(
    file_path,
    encoding='utf-8',
    usecols=['用户ID', '性别', '年龄', '城市', '消费金额(元)', '是否付费']  # 仅读核心列
)
cols_time = time.time() - start_time
cols_memory = df_cols.memory_usage(deep=True).sum() / 1024 / 1024
print("\n2.1 仅读核心列优化：")
print(f"读取耗时：{cols_time:.4f} 秒（节省{(1-cols_time/basic_time)*100:.2f}%）")
print(f"内存占用：{cols_memory:.4f} MB（节省{(1-cols_memory/basic_memory)*100:.2f}%）")


2.1 仅读核心列优化：
读取耗时：0.0038 秒（节省-2.92%）
内存占用：0.2370 MB（节省53.16%）


In [24]:
dtype_spec = {
    '用户ID': 'int16',
    '年龄': 'int8',
    '性别': 'category',
    '城市': 'category',
    '消费金额(元)': 'float32',
    '是否付费': 'object'
}

start_time = time.time()
df_dtype = pd.read_csv(
    file_path,
    encoding='utf-8',
    usecols=['用户ID', '性别', '年龄', '城市', '消费金额(元)', '是否付费'],
    dtype=dtype_spec
)
# 转换布尔字段
df_dtype['是否付费'] = df_dtype['是否付费'].map({'是': True, '否': False}).astype('bool')

dtype_time = time.time() - start_time
dtype_memory = df_dtype.memory_usage(deep=True).sum() / 1024 / 1024

print("\n2.2 预设类型+列筛选优化：")
print(f"读取耗时：{dtype_time:.4f} 秒（节省{(1-dtype_time/basic_time)*100:.2f}%）")
print(f"内存占用：{dtype_memory:.4f} MB（节省{(1-dtype_memory/basic_memory)*100:.2f}%）")


2.2 预设类型+列筛选优化：
读取耗时：0.0036 秒（节省2.78%）
内存占用：0.0111 MB（节省97.81%）


In [25]:
print("\n2.3 分块读取（超大文件专用）：")
chunk_size = 100  # 每块100行
chunks = []
start_time = time.time()

# 分块读取并处理
for chunk in pd.read_csv(file_path, encoding='utf-8', chunksize=chunk_size):
    # 块内优化：筛选列+改类型
    chunk = chunk[['用户ID', '性别', '消费金额(元)']]
    chunk['用户ID'] = chunk['用户ID'].astype('int16')
    chunk['性别'] = chunk['性别'].astype('category')
    chunks.append(chunk)
# 合并分块
df_chunk = pd.concat(chunks, ignore_index=True)
chunk_time = time.time() - start_time
chunk_memory = df_chunk.memory_usage(deep=True).sum() / 1024 / 1024
print(f"分块读取耗时：{chunk_time:.4f} 秒")
print(f"合并后内存：{chunk_memory:.4f} MB")
print(f"分块读取优势：内存峰值仅单块大小，适合超大型文件")


2.3 分块读取（超大文件专用）：
分块读取耗时：0.0187 秒
合并后内存：0.0116 MB
分块读取优势：内存峰值仅单块大小，适合超大型文件


In [26]:
df_dtype

,用户ID,性别,年龄,城市,消费金额(元),是否付费
0,1001,男,25,北京,2999.000000,True
1,1002,女,32,上海,199.500000,True
2,1003,男,28,广州,318.799988,False
3,1004,女,40,深圳,99.000000,False
4,1005,男,35,北京,2999.000000,True
...,...,...,...,...,...,...
1075,1006,女,29,上海,299.899994,True
1076,1007,男,45,广州,129.800003,False
1077,1008,女,27,深圳,799.500000,True
1078,1009,男,33,北京,599.000000,False


In [27]:
import pandas as pd
import time

# 🔥 直接用 fastparquet 引擎，彻底避开 PyArrow 冲突
# 写入（核心：engine="fastparquet"）
df_dtype.to_parquet(
    'read_optimize.parquet',
    index=False,
    engine="fastparquet"  # 👈 就改这一句
)

# 读取（同样用 fastparquet）
start_time = time.time()
df_parquet = pd.read_parquet('read_optimize.parquet', engine="fastparquet")
parquet_time = time.time() - start_time
parquet_memory = df_parquet.memory_usage(deep=True).sum() / 1024 / 1024

print("\n2.4 转换为Parquet格式读取：")
print(f"读取耗时：{parquet_time:.4f} 秒（节省{(1-parquet_time/basic_time)*100:.2f}%）")
print(f"内存占用：{parquet_memory:.4f} MB")


2.4 转换为Parquet格式读取：
读取耗时：0.0087 秒（节省-135.53%）
内存占用：0.0108 MB


In [28]:
df_skip = pd.read_csv(
    file_path,
    encoding='utf-8',
    skip_blank_lines=True,  # 跳过空行
    comment='#'  # 跳过#开头的注释行
)

In [29]:
df_na = pd.read_csv(
    file_path,
    encoding='utf-8',
    na_values=['无', '未知'],  # 将"无"/"未知"识别为NaN
    keep_default_na=False  # 仅识别指定缺失值
)

In [30]:
optimize_summary = pd.DataFrame(
    {
        '读取方式': ['基础读取', '仅读核心列', '预设类型+列筛选', '分块读取', 'Parquet读取'],
        '耗时(秒)': [basic_time, cols_time, dtype_time, chunk_time, parquet_time],
        '内存(MB)': [basic_memory, cols_memory, dtype_memory, chunk_memory, parquet_memory]
    }
).round(4)
print("\n4. 优化效果汇总：")
print(optimize_summary)


4. 优化效果汇总：
        读取方式   耗时(秒)  内存(MB)
0       基础读取  0.0037  0.5060
1      仅读核心列  0.0038  0.2370
2   预设类型+列筛选  0.0036  0.0111
3       分块读取  0.0187  0.0116
4  Parquet读取  0.0087  0.0108
